# RMSearch Test Pipeline

This notebook mirrors the main RMSearch training flow (prepare data → generate tag graph → build dataset → train → convert → retrieval evaluation) while relying on the vLLM worker pool implementation in `rmsearch.utils.vllm_generate` instead of the asynchronous `AllRequests` helper.

In [3]:
!pip install ..

Processing /workspace/RMSearch
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-4.57.0-py3-none-any.whl.metadata (41 kB)
  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached peft-0.17.1-py3-none-any.whl.metadata (14 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.6-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached sentence_transformers-5.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached rich-14.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached safetenso

In [1]:
!pytest -q

/bin/bash: line 1: pytest: command not found


## Configuration

In [4]:
import json
import os
from pathlib import Path

import multiprocessing as mp
mp.set_start_method("spawn", force=True)

WORKING_DIR = Path("/workspace/RMS_exp")
DATA_NAME = "smollm-corpus"
LOCAL_DATA_DIR = Path("./data") / DATA_NAME
WORKING_DATA_DIR = WORKING_DIR / "data" / DATA_NAME

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
WORKING_DATA_DIR.mkdir(parents=True, exist_ok=True)

GENERATION_MODEL = os.environ.get("VLLM_MODEL", "/workspace/qwen7b")
EMBED_MODEL = os.environ.get("EMB_MODEL", "intfloat/e5-mistral-7b-instruct")
REWARD_BASE_MODEL = os.environ.get("RM_BASE_MODEL", "/workspace/llama3b-rm")
CONVERTED_MODEL_DIR = WORKING_DIR / "models" / "reward-converted"

GENERATOR_SETTINGS = {
    "tensor_parallel_size": 1,
    "num_instances": 1,
    "device_groups": None,
    "llm_kwargs": {"max_model_len": 4096, "max_num_seqs": 64, "gpu_memory_utilization": 0.9},
    "batch_size": 8,
    "timeout_s": None,
}
EMBED_SETTINGS = {
    "tensor_parallel_size": 1,
    "num_instances": 1,
    "device_groups": None,
    "llm_kwargs": {"max_model_len": 2048, "max_num_seqs": 64, "gpu_memory_utilization": 0.9},
    "batch_size": 32,
    "timeout_s": None,
}

print(f"Data dir: {LOCAL_DATA_DIR}")
print(f"Working dir: {WORKING_DIR}")

Data dir: data_test/smollm-corpus
Working dir: /workspace/RMS_exp


## Prepare Data

In [5]:
import shutil

from rmsearch.train import process_data

processed_dir = process_data(
    "HuggingFaceTB/smollm-corpus",
    output_dir=LOCAL_DATA_DIR,
    n_sample_train=100_000,
    n_sample_test=8_000,
    n_small_sample=10_000,
    random_seed=42,
)

print(f"Processed dataset saved to {processed_dir}")

for filename in ["df.csv", "df_small.csv", "dataset_dict.json"]:
    src = LOCAL_DATA_DIR / filename
    if src.exists():
        shutil.copy2(src, WORKING_DATA_DIR / filename)

INFO 10-08 00:33:44 [__init__.py:216] Automatically detected platform cuda.


Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

ValueError: At least one data file must be specified, but got data_files=None

In [ ]:
import pandas as pd

df_small = pd.read_csv(LOCAL_DATA_DIR / "df_small.csv")
display(df_small.head())
print(f"Sample size: {len(df_small)}")

## Generate Tag Graph v2

In [ ]:
from vllm import SamplingParams

from rmsearch.tree import generate_tag
from rmsearch.utils.vllm_generate import build_llm

keys = df_small["text"].astype(str).tolist()

tag_pool = build_llm(
    model_name=GENERATION_MODEL,
    tensor_parallel_size=GENERATOR_SETTINGS["tensor_parallel_size"],
    num_instances=GENERATOR_SETTINGS["num_instances"],
    device_groups=GENERATOR_SETTINGS["device_groups"],
    **GENERATOR_SETTINGS["llm_kwargs"],
)

try:
    tag_recs = generate_tag(
        keys,
        model_name=GENERATION_MODEL,
        model=tag_pool,
        sampling_params=SamplingParams(max_tokens=128, temperature=0.3, top_p=0.9),
        worker_batch_size=GENERATOR_SETTINGS["batch_size"],
        timeout_s=GENERATOR_SETTINGS["timeout_s"],
    )
finally:
    tag_pool.close()

with (WORKING_DATA_DIR / "tag_recs.json").open("w", encoding="utf-8") as handle:
    json.dump(tag_recs, handle, ensure_ascii=False, indent=2)

len(tag_recs)

In [ ]:
import torch

from rmsearch.tree import embed_tags
from rmsearch.utils.vllm_embed import build_embedding_model

embed_pool = build_embedding_model(
    model_name=EMBED_MODEL,
    tensor_parallel_size=EMBED_SETTINGS["tensor_parallel_size"],
    num_instances=EMBED_SETTINGS["num_instances"],
    device_groups=EMBED_SETTINGS["device_groups"],
    **EMBED_SETTINGS["llm_kwargs"],
)

try:
    tag_embeddings, tag_meta = embed_tags(
        tag_recs,
        embed_model_name=EMBED_MODEL,
        pool=embed_pool,
        worker_batch_size=EMBED_SETTINGS["batch_size"],
        timeout_s=EMBED_SETTINGS["timeout_s"],
    )
finally:
    embed_pool.close()

torch.save(tag_embeddings, WORKING_DATA_DIR / "tag_embeddings.pt")
with (WORKING_DATA_DIR / "tag_meta.json").open("w", encoding="utf-8") as handle:
    json.dump(tag_meta, handle, ensure_ascii=False, indent=2)

tag_embeddings.shape

In [ ]:
from typing import Dict, List

from rmsearch.tree import HierarchicalKMeans, build_representative_tags

embedding_lookup: Dict[int, str] = {
    idx: tag_recs[key_id]["tags"][tag_idx]
    for idx, (key_id, tag_idx) in enumerate(tag_meta)
}

def attach_tags(nodes: List[Dict[str, any]]) -> List[Dict[str, any]]:
    for node in nodes:
        tag_ids = node.get("tag_ids", [])
        node["tags"] = [embedding_lookup.get(i, "") for i in tag_ids if i in embedding_lookup]
        if "children" in node:
            attach_tags(node["children"])
    return nodes

kmeans = HierarchicalKMeans(n_clusters=8, max_leaf_size=60, random_state=42)
kmeans.fit(tag_embeddings)

tag_tree = kmeans.leaf_members_json()
tag_tree = attach_tags(tag_tree)

tag_tree = build_representative_tags(
    tag_tree,
    model_name=GENERATION_MODEL,
    tensor_parallel_size=GENERATOR_SETTINGS["tensor_parallel_size"],
    num_instances=GENERATOR_SETTINGS["num_instances"],
    device_groups=GENERATOR_SETTINGS["device_groups"],
    worker_batch_size=GENERATOR_SETTINGS["batch_size"],
    timeout_s=GENERATOR_SETTINGS["timeout_s"],
    n_tag_sample=6,
)

with (WORKING_DATA_DIR / "tag_tree_recs.json").open("w", encoding="utf-8") as handle:
    json.dump(tag_tree, handle, ensure_ascii=False, indent=2)

len(tag_tree)

## Make Dataset

In [ ]:
from transformers import AutoTokenizer

from rmsearch.train import make_queries

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL, padding_side="left")
query_pool = build_llm(
    model_name=GENERATION_MODEL,
    tensor_parallel_size=GENERATOR_SETTINGS["tensor_parallel_size"],
    num_instances=GENERATOR_SETTINGS["num_instances"],
    device_groups=GENERATOR_SETTINGS["device_groups"],
    **GENERATOR_SETTINGS["llm_kwargs"],
)

query_sampling = SamplingParams(max_tokens=512, temperature=0.0, top_p=0.9)

def run_with_pool(prompts):
    return query_pool.generate(
        prompts,
        sampling_params=query_sampling,
        batch_size=GENERATOR_SETTINGS["batch_size"],
        timeout_s=GENERATOR_SETTINGS["timeout_s"],
    )

try:
    query_dict = make_queries(
        df_small["text"].astype(str).tolist(),
        tokenizer=tokenizer,
        request_func=run_with_pool,
    )
finally:
    query_pool.close()

with (WORKING_DATA_DIR / "query_dict.json").open("w", encoding="utf-8") as handle:
    json.dump(query_dict, handle, ensure_ascii=False, indent=2)

len(query_dict)

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import torch

question_records = []
for key, meta in query_dict.items():
    idx = int(key)
    for question in meta.get("questions", []):
        question_records.append({"query": question, "correct_id": idx})

questions = [row["query"] for row in question_records]
if not questions:
    raise ValueError("make_queries returned no questions; adjust prompts or data sample size.")

st_model = SentenceTransformer(EMBED_MODEL).to("cuda")
sentence_embeddings = st_model.encode(df_small["text"].astype(str).tolist(), convert_to_tensor=True, show_progress_bar=True)
query_embeddings = st_model.encode(questions, convert_to_tensor=True, show_progress_bar=True)

scores = cos_sim(query_embeddings, sentence_embeddings)
TOP_K = 10
relevant_sentences = []
for row_idx, record in enumerate(question_records):
    values, indices = torch.topk(scores[row_idx], k=min(TOP_K, scores.shape[1]))
    relevant_sentences.append(
        {
            "query": record["query"],
            "query_id": row_idx,
            "correct_id": record["correct_id"],
            "keys": [
                {
                    "key_id": int(idx),
                    "key": df_small.iloc[int(idx)]["text"],
                    "score": float(score),
                }
                for idx, score in zip(indices.tolist(), values.tolist())
            ],
        }
    )

with (WORKING_DATA_DIR / "sentences_relevant_to_questions.json").open("w", encoding="utf-8") as handle:
    json.dump(relevant_sentences, handle, ensure_ascii=False, indent=2)

len(relevant_sentences)

In [ ]:
from itertools import combinations

from rmsearch.train.utils import extract_int, extract_text

judge_pool = build_llm(
    model_name=GENERATION_MODEL,
    tensor_parallel_size=GENERATOR_SETTINGS["tensor_parallel_size"],
    num_instances=GENERATOR_SETTINGS["num_instances"],
    device_groups=GENERATOR_SETTINGS["device_groups"],
    **GENERATOR_SETTINGS["llm_kwargs"],
)
judge_sampling = SamplingParams(max_tokens=512, temperature=0.0, top_p=0.9)

judge_prompts = []
judge_meta = []
for sentence_dict in relevant_sentences:
    candidates = sentence_dict["keys"][:3]
    if len(candidates) < 2:
        continue
    for idx_a, idx_b in combinations(range(len(candidates)), 2):
        sent_a = candidates[idx_a]
        sent_b = candidates[idx_b]
        prompt = tokenizer.apply_chat_template(
            [
                {
                    "role": "system",
                    "content": (
                        "You are a brilliant judge who decides which text is more relevant to a given query. "
                        "Output format: <ID>1</ID> or <ID>2</ID>."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Query: {sentence_dict['query']}

"
                        f"Sentence 1:
'''{sent_a['key']}'''

"
                        f"Sentence 2:
'''{sent_b['key']}'''

"
                        "Return the id of the more relevant sentence enclosed within <ID></ID>. "
                        "If both are equally relevant or irrelevant, respond with <ID>-1</ID>."
                    ),
                },
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        judge_prompts.append(prompt)
        judge_meta.append({
            "sentence_ids": [sent_a["key_id"], sent_b["key_id"]],
            "question": sentence_dict["query"],
        })

judge_outputs = judge_pool.generate(
    judge_prompts,
    sampling_params=judge_sampling,
    batch_size=GENERATOR_SETTINGS["batch_size"],
    timeout_s=GENERATOR_SETTINGS["timeout_s"],
)
judge_pool.close()

judgements = []
for meta_record, output_text in zip(judge_meta, judge_outputs):
    record = dict(meta_record)
    record["output"] = output_text
    judgements.append(record)

with (WORKING_DATA_DIR / "judge_results.json").open("w", encoding="utf-8") as handle:
    json.dump(judgements, handle, ensure_ascii=False, indent=2)

len(judgements)

In [ ]:
from rmsearch.train.lora_example import make_dataset_list

sentences = df_small["text"].astype(str).tolist()
dataset_list = make_dataset_list(judgements, sentences=sentences)

dataset_list_path = WORKING_DIR / "exp" / "dataset_list.json"
dataset_list_path.parent.mkdir(parents=True, exist_ok=True)
dataset_list_path.write_text(json.dumps(dataset_list, ensure_ascii=False, indent=2))

len(dataset_list)

## Train

In [ ]:
from rmsearch.train.lora_example import train_reward_model

RUN_TRAINING = False  # Flip to True when you are ready to fine-tune the reward model.

if RUN_TRAINING:
    train_reward_model(
        dataset_list,
        model_name=REWARD_BASE_MODEL,
        num_gpus=2,
        output_dir=WORKING_DIR / "exp" / "model1",
        base_dir=WORKING_DIR / "exp",
    )
else:
    print("Training skipped (set RUN_TRAINING=True to enable).")

## Model Conversion

In [ ]:
from rmsearch.train.utils import convert_model

checkpoint_dir = WORKING_DIR / "exp" / "model1"
if checkpoint_dir.exists():
    convert_model(
        base_model_name=REWARD_BASE_MODEL,
        checkpoint_path=str(checkpoint_dir),
        output_dir=CONVERTED_MODEL_DIR,
    )
else:
    print(f"Skip conversion: {checkpoint_dir} does not exist yet.")

## Evaluation (Retrieval)

In [ ]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)

from rmsearch.evaluation.retrieval import retrieval_evaluation
from rmsearch.utils.vllm_reward2 import build_llm, search

if CONVERTED_MODEL_DIR.exists():
    sentences = df_small["text"].astype(str).tolist()
    queries = [row["query"] for row in question_records]
    correct_ids = [row["correct_id"] for row in question_records]

    if not queries:
        raise ValueError("No queries available for evaluation.")

    rm_model = build_llm(
        model_name=str(CONVERTED_MODEL_DIR),
        tensor_parallel_size=GENERATOR_SETTINGS["tensor_parallel_size"],
        num_instances=GENERATOR_SETTINGS["num_instances"],
        device_groups=GENERATOR_SETTINGS["device_groups"],
        max_model_len=2500,
        max_num_seqs=64,
        gpu_memory_utilization=0.9,
        runner="pooling",
    )
    tokenizer = rm_model.tokenizer

    def llm_template_func(row):
        message = [
            {
                "role": "user",
                "content": (
                    "Give me relevance score between

"
                    f"Query:{row['query']}

"
                    f"Sentence:{row['key']}"
                ),
            }
        ]
        prompt = tokenizer.apply_chat_template(message, tokenize=False)
        return prompt

    def reward_search(requests):
        if not requests:
            return []
        topk = max(req.get("k", 10) for req in requests)
        return search(
            rm_model,
            requests,
            llm_template_func,
            topk=topk,
            batch_size=GENERATOR_SETTINGS["batch_size"],
            timeout_s=GENERATOR_SETTINGS["timeout_s"] or 10_000,
        )

    try:
        evaluation = retrieval_evaluation(
            queries,
            sentences,
            tag_tree,
            search_fn=reward_search,
            k_tag=2,
            k_key=10,
            correct_ids=correct_ids,
        )
    finally:
        rm_model.close()

    with (WORKING_DATA_DIR / "relevance_dict.json").open("w", encoding="utf-8") as handle:
        json.dump(evaluation, handle, ensure_ascii=False, indent=2)

    len(evaluation)
else:
    print("Skip evaluation: converted reward model not found.")